# Módulo 47 · DL III: API funcional, HOG y modelos con varias entradas

Código del módulo para ejecutar en **Google Colab**, donde TensorFlow y PyTorch ya vienen instalados. Para entrenar más rápido activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Las explicaciones, los ejercicios y las partes que funcionan en el navegador están en el módulo 47 de la web.

In [ ]:
# Solo si lo ejecutas en tu ordenador (en Colab ya está todo instalado):
# !pip install tensorflow scikit-image pydot

### Keras: API funcional con dos entradas (Colab)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate
from skimage.feature import hog
from sklearn.preprocessing import normalize

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train_flat = x_train.reshape(len(x_train), -1).astype("float32") / 255     # mejor que escribir 60000 a mano
x_test_flat = x_test.reshape(len(x_test), -1).astype("float32") / 255

def calc_hog_features(X, pixels_per_cell=(4, 4)):
    return np.array([hog(img, orientations=8, pixels_per_cell=pixels_per_cell, cells_per_block=(1, 1)) for img in X])

hog_train = normalize(calc_hog_features(x_train.astype("float32")))    # (60000, 392): 7x7 celdas x 8 orientaciones
hog_test = normalize(calc_hog_features(x_test.astype("float32")))

# Modelo 1: píxeles
entrada1 = Input(shape=(784,))
oculta1 = Dense(20, activation="relu")(entrada1)
model1 = Model(entrada1, Dense(10, activation="softmax")(oculta1))

# Modelo 2: HOG
entrada2 = Input(shape=(392,))
oculta2 = Dense(10, activation="sigmoid")(entrada2)
model2 = Model(entrada2, Dense(10, activation="softmax")(oculta2))

for modelo, (Xtr, Xte) in [(model1, (x_train_flat, x_test_flat)), (model2, (hog_train, hog_test))]:
    modelo.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    modelo.fit(Xtr, y_train, batch_size=32, epochs=10, validation_split=0.15, verbose=0)
    print("Test accuracy:", modelo.evaluate(Xte, y_test, verbose=0)[1])

# Modelo 3: las dos entradas, concatenando las capas ocultas de los modelos anteriores
h = Concatenate()([oculta1, oculta2])
model3 = Model(inputs=[entrada1, entrada2], outputs=Dense(10, activation="softmax")(h))
model3.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model3.fit([x_train_flat, hog_train], y_train, batch_size=32, epochs=10, validation_split=0.15, verbose=0)
print("Modelo combinado, test accuracy:", model3.evaluate([x_test_flat, hog_test], y_test, verbose=0)[1])
tf.keras.utils.plot_model(model3, show_shapes=True, show_layer_names=True)    # necesita pydot y graphviz